# HCP Grain Boundary Generator — Getting Started

This notebook demonstrates how to:
1. Enumerate available CSL grain boundaries for any c/a ratio
2. Query and filter by Sigma, rotation axis, or angle
3. Build twist and tilt grain boundary structures with ASE
4. Visualise and export the structures

In [ ]:
from math import sqrt
from hcp_gb_generator import (
    enumerate_hcp_csl,
    enumerate_0001_csl,
    find_csl,
    print_csl_table,
    to_dataframe,
    build_gb,
    build_twist_gb,
    build_tilt_gb,
    csl_slab_directions,
)

## 1. Enumerate CSL grain boundaries

### 1a. [0001]-axis GBs (c/a independent, fast)

These are basal-plane twist boundaries.  The available Sigma values
depend only on the 2D triangular lattice geometry and are the same
for **any** HCP material.

In [ ]:
res_0001 = enumerate_0001_csl(sigma_max=50)
print_csl_table(res_0001)

### 1b. Full enumeration (all axes, c/a dependent)

Tilt GBs with rotation axes in the basal plane depend on the c/a ratio.
Common values:
- Ti:  c/a ≈ 1.587
- Mg:  c/a ≈ 1.624
- Ideal HCP: c/a = sqrt(8/3) ≈ 1.633
- Zn:  c/a ≈ 1.856

In [ ]:
IDEAL_CA = sqrt(8 / 3)

all_results = enumerate_hcp_csl(
    ca_ratio=IDEAL_CA,
    sigma_max=25,
    max_idx=3,       # max Miller index for axis search (3 is fast, 9 for thorough)
)

print(f"Found {len(all_results)} CSL grain boundaries")
print_csl_table(all_results)

### 1c. As a pandas DataFrame

In [ ]:
df = to_dataframe(all_results)
df.head(15)

## 2. Query and filter

### 2a. Find all GBs for a specific Sigma

In [ ]:
sigma7 = find_csl(all_results, sigma=7)
for r in sigma7:
    print(f"  Sigma={r['sigma']}  disorientation={r['disorientation_angle']:.2f}  axis={r['axis_miller']}")

### 2b. Filter by rotation axis family

Use 4-index Miller-Bravais notation.  The filter is hex-symmetry-aware,
so `[1,0,-1,0]` matches all `<10-10>` family members.

In [ ]:
# [0001] axis — basal twist boundaries
print("=== [0001] axis ===")
for r in find_csl(all_results, axis_miller_bravais=[0, 0, 0, 1], ca_ratio=IDEAL_CA):
    print(f"  Sigma={r['sigma']:>3}  disorientation={r['disorientation_angle']:>7.2f}")

# [10-10] axis — prism tilt boundaries
print("\n=== [10-10] axis ===")
for r in find_csl(all_results, axis_miller_bravais=[1, 0, -1, 0], ca_ratio=IDEAL_CA):
    print(f"  Sigma={r['sigma']:>3}  disorientation={r['disorientation_angle']:>7.2f}")

# [11-20] axis — a-direction tilt boundaries
print("\n=== [11-20] axis ===")
for r in find_csl(all_results, axis_miller_bravais=[1, 1, -2, 0], ca_ratio=IDEAL_CA):
    print(f"  Sigma={r['sigma']:>3}  disorientation={r['disorientation_angle']:>7.2f}")

### 2c. Filter by angle range

In [ ]:
low_angle = find_csl(all_results, angle_min=10, angle_max=30)
print(f"Found {len(low_angle)} GBs between 10-30 degrees:")
for r in low_angle:
    print(f"  Sigma={r['sigma']:>3}  disorientation={r['disorientation_angle']:>7.2f}  axis={r['axis_miller']}")

### 2d. Generate on-the-fly for a different c/a ratio

In [ ]:
# No pre-computed results needed — find_csl generates them
ti_sigma7 = find_csl(ca_ratio=1.587, sigma=7, sigma_max=10)
print(f"Ti (c/a=1.587): {len(ti_sigma7)} Sigma=7 boundaries")
for r in ti_sigma7:
    print(f"  disorientation={r['disorientation_angle']:.2f}  axis={r['axis_miller']}")

## 3. Build grain boundary structures

### 3a. [0001] Twist grain boundary

Both grains share the (0001) surface.  The upper grain is rotated
in-plane by the CSL angle.

In [ ]:
# Titanium lattice parameters
a_Ti, c_Ti = 2.95, 4.68

# Find Sigma=7 [0001] twist
rec = find_csl(ca_ratio=c_Ti / a_Ti, sigma=7,
               axis_miller_bravais=[0, 0, 0, 1], sigma_max=10)[0]

# Build the bicrystal
twist_gb = build_twist_gb(
    rec,
    element="Ti",
    a=a_Ti,
    c=c_Ti,
    n_layers=4,
    gap=0.5,
)

print(f"Sigma = {twist_gb.info['sigma']}")
print(f"Disorientation = {twist_gb.info['disorientation_angle']:.2f} deg")
print(f"Atoms = {len(twist_gb)}")
print(f"Cell  = {twist_gb.cell.lengths().round(2)}")
print(f"Grains: {sorted(set(twist_gb.arrays['grain_id']))}")

### 3b. Symmetric tilt grain boundary

Rotation axis lies in the GB plane.  The two grains are
mirror images across the boundary.

In [ ]:
# Tilt GBs are c/a-dependent — use ideal c/a where many are known
# (Ti's non-ideal c/a produces fewer tilt CSLs at low Sigma)
ca_ideal = sqrt(8 / 3)
a_ideal, c_ideal = 2.95, 2.95 * ca_ideal  # scale to Ti-like size

all_res = enumerate_hcp_csl(ca_ratio=ca_ideal, sigma_max=20, max_idx=3)
rec_tilt = find_csl(all_res, sigma=17,
                    axis_miller_bravais=[1, 1, -2, 0],
                    ca_ratio=ca_ideal)[0]

tilt_gb = build_tilt_gb(
    rec_tilt,
    element="Ti",
    a=a_ideal,
    c=c_ideal,
    n_layers=4,
    gap=0.5,
)

print(f"Sigma = {tilt_gb.info['sigma']}")
print(f"Disorientation = {tilt_gb.info['disorientation_angle']:.2f} deg")
print(f"Atoms = {len(tilt_gb)}")
print(f"Upper dirs = {tilt_gb.info['upper_dirs']}")
print(f"Lower dirs = {tilt_gb.info['lower_dirs']}")

### 3c. Auto-dispatch with `build_gb`

`build_gb` detects whether the rotation axis is [0001] (twist)
or in-plane (tilt) and calls the appropriate builder.

In [ ]:
# Twist — automatically detected
twist = build_gb(rec, element="Ti", a=a_Ti, c=c_Ti, n_layers=3)
print(f"Twist: {len(twist)} atoms")

# Tilt — automatically detected (using ideal c/a params from above)
tilt = build_gb(rec_tilt, element="Ti", a=a_ideal, c=c_ideal, n_layers=3)
print(f"Tilt:  {len(tilt)} atoms")

### 3d. Get slab directions for use with other GB builders

If you have an existing GB construction pipeline (e.g. `HCPGBGenerator`),
you can use `csl_slab_directions` to get the 4-index Miller-Bravais
direction triples.

In [ ]:
upper_dirs, lower_dirs = csl_slab_directions(rec_tilt, ca=ca_ideal)
print(f"Upper grain directions (4-index): {upper_dirs}")
print(f"Lower grain directions (4-index): {lower_dirs}")
print()
print("These can be passed to ASE's HexagonalClosedPacked(directions=...)")
print("or to HCPGBGenerator.from_basal_directions()")

## 4. Export structures

The returned `ase.Atoms` objects work with all ASE I/O formats.

In [ ]:
from ase.io import write

# POSCAR (VASP)
write("POSCAR_twist_S7", twist_gb, format="vasp")

# LAMMPS data file
write("twist_S7.lammps", twist_gb, format="lammps-data")

# XYZ (for visualisation)
write("twist_S7.xyz", twist_gb)

print("Exported: POSCAR_twist_S7, twist_S7.lammps, twist_S7.xyz")

## 5. Comparing materials

The [0001] twist CSLs are universal, but tilt GBs differ between materials.

In [ ]:
materials = {
    "Ti":       {"ca": 1.587, "a": 2.95,  "c": 4.68},
    "Mg":       {"ca": 1.624, "a": 3.21,  "c": 5.21},
    "Ideal":    {"ca": sqrt(8/3), "a": 1.0, "c": sqrt(8/3)},
}

for name, params in materials.items():
    res = enumerate_hcp_csl(ca_ratio=params["ca"], sigma_max=25, max_idx=3)
    sigmas = sorted(set(r["sigma"] for r in res))
    print(f"{name:>7} (c/a={params['ca']:.3f}): {len(res)} GBs, "
          f"Sigmas = {sigmas}")